# **RANDOM FOREST**
*Bagging-based technique*
<br>
*Collection of decision trees*

Random Forest = "Random" (randomness in row/column sampling) + "Forest" (collection of many trees)

The overall technique behind this is called **Bagging = Bootstrap Aggregating**
- **Bootstrap** = the random sampling part (creating different data subsets)
- **Aggregating** = the combining/voting part (combining all trees' predictions)

### **What is Bootstrap?**
Suppose you have 1000 rows.
You have many base models (decision trees, since we're using random forest).
Instead of giving each tree the whole dataset, you give it a random **sample** of the data.

*DTx = Decision Tree x*

**Three types of sampling:**
1) **Row sampling** - randomly select some number of rows and give them to DT1.
   - **With replacement** - a row can be picked more than once (duplicates possible) - this is what bagging/random forest actually uses.
   - **Without replacement** - each row picked only once, no duplicates.
2) **Column/feature sampling** - same idea, but randomly selecting which columns/features each tree gets to see (not all columns every time).
3) **Combination** - both row and column sampling done together (this is exactly what random forest does — each tree gets a random subset of rows AND a random subset of columns).

That whole process - creating these random samples for each tree - is called **Bootstrap**.

### **What is Aggregation?**
Suppose a new point comes in and you need to predict its class.
You give this point to all your base models (DT1, DT2, DT3, ...).
Each tree makes its own prediction.

- **For classification** -> take the majority vote (e.g., if most trees predict "1", the final prediction is "1").
- **For regression** -> take the **average** (mean) of all the base models' predictions instead of a vote.

That combining step - voting for classification, averaging for regression — is called **Aggregation**.

## **`Bias:`**
Giving wrong answers even on the training dataset - meaning the model doesn't understand the training data properly. This is **high bias**.

## **`Variance:`**
The model "rote learns" the training dataset - performs great on training data, but whenever the data changes even slightly (new/test data), it gives wrong answers. This is **high variance**.

So we want **low bias and low variance** - but there's an inherent tradeoff between the two (as one goes down, the other tends to go up). This is called the **bias-variance tradeoff**.

Most ML algorithms fall into one of two problematic classes:

**→ Low Bias, High Variance (LB & HV)**
Performs well on training data, but not accurate on test data (overfitting).
Examples: Fully grown Decision Tree, SVM, KNN (in certain cases)

**→ High Bias, Low Variance (HB & LV)**
Doesn't fit the training data that well, but is more stable/consistent (underfitting).
Examples: Linear Regression, Logistic Regression (in certain cases)

**We rarely get the ideal combination we actually want: Low Bias + Low Variance.**

**That's why Random Forest exists** - it takes a Low Bias, High Variance base model (fully-grown decision trees) and turns it into a Low Bias, Low Variance overall model. It keeps the bias low (since individual trees still fit the data well) while reducing the variance significantly (by combining many trees).

### **How does it work?**

Suppose you have 10,000 rows, and you distribute this data to your base models (decision trees) using **row sampling** (bootstrap, with replacement). Since sampling is with replacement, each tree only sees a random subset of the data, and the rows left out for a given tree (called **out-of-bag rows**) differ from tree to tree.

Because each tree sees a *different* random subset, each individual tree ends up slightly different - and each one alone still has high variance (sensitive to its particular sample).

But here's the key: since the sampling is *random and independent* for every tree, their individual errors/mistakes tend to be different from each other, not the same mistakes repeated. So when you **combine (aggregate)** all these trees' predictions - via voting or averaging — the individual high-variance errors cancel out, and the overall combined model ends up with much lower variance than any single tree.

**So: bias stays low (each tree still fits data well) + variance drops (errors cancel out when combined) = Random Forest achieves the Low Bias, Low Variance combination that a single tree alone can't.**

## **IMP: Bagging vs Random Forest**

In **Bagging**, you can use different-different base models - Decision Tree, KNN, SVM, etc.

In **Random Forest**, you can only use Decision Tree as the base model.

**What if...**
I select Decision Tree as the base model for a bagging ensemble - does that now become Random Forest? 

**Answer: No!**

A key difference still exists: **feature/column sampling**

**Meaning:**
- In **Bagging**, column sampling happens at the **tree level** - meaning the set of columns for a tree is decided once, before building that tree, and stays fixed for all its splits.
- In **Random Forest**, column sampling happens at the **node level** - meaning a *new* random subset of columns is chosen every single time a node is about to split (more randomness, since every split gets its own fresh random column subset).

This extra layer of randomness (node-level vs tree-level column sampling) is exactly why Random Forest usually performs better than plain Bagging with decision trees — more randomness across trees means their individual errors are even more different from each other, so they cancel out more effectively when combined (lower variance overall, per what we learned in the bias-variance section).

## **Random Forest Hyperparameters**

**`n_estimators`:** Decides how many decision trees your Random Forest is built with (e.g., n_estimators=100 means 100 trees are trained and their results combined). More trees generally = more stable/accurate, but slower to train — there's usually a point of diminishing returns.

**`max_features`:** If you have n columns in your dataset, this decides how many columns are randomly considered at each split. (Common defaults: `sqrt(n)` for classification, `n/3` for regression — but you can set any number.)

**`bootstrap`:** Whether row sampling is done with or without replacement. If `True` (default), sampling happens **with replacement** (duplicates possible — this is the actual bootstrap behavior). If `False`, sampling is done **without replacement** (each row used only once per tree — technically this turns it into "Pasting" instead of true bagging).

**`max_samples`:** How many rows (or what percentage of rows) are given to each individual decision tree. Common practice is to keep this around 50-75% of the total data — giving each tree enough data to learn well, while still keeping enough randomness between trees for the variance-reduction benefit to work.

## **Out-of-Bag (OOB) Samples**

**Out-of-bag samples** are the subset of the original training data that were **not** included in the bootstrap sample used to train a particular base model (like a decision tree) in an ensemble.

This concept arises from the bootstrapping process:

- **Bootstrapping:** To build an ensemble model like Random Forest, you create multiple training subsets by sampling the original data **with replacement**. This means some data points get selected multiple times, while others aren't selected at all for a given tree.

- **In-Bag Samples:** The data points that WERE selected for a specific subset (used to train a specific tree) are called the **in-bag samples**.

- **Out-of-Bag Samples (OOB):** The data points that were NOT selected for that specific subset are the **out-of-bag samples**, for that particular tree.

- On average, for a sufficiently large dataset, roughly **~37%** of the original data points get left out as OOB samples for any given bootstrap sample. *(This comes from the math of sampling with replacement: as sample size grows, the probability a given row is never picked converges to 1/e ≈ 0.368.)*

### **Why OOB samples matter:**

They provide a **built-in, unbiased validation set** for each individual tree in the ensemble — without needing a separate train/validation split or cross-validation.

- **Internal Validation:** Each tree is tested on the OOB samples it never saw during its own training.
- **OOB Error/Score:** The overall OOB score for the ensemble is calculated by aggregating predictions for each original data point using only the trees for which that point was OOB (i.e., that tree never trained on it). This aggregated OOB error gives a reliable estimate of how well the model generalizes to unseen data.

# Feature importance
How is this feature importand for us so that we can drop out remaining feature
<br>
Random forest can detect which features are important and which are not
<br>
We can use feature importance by using random forest, decision trees as well as from bagging


# `CODE PART IN TELCO CHURN FILE`